# Migrate Volume Files to Production Hierarchy

This notebook migrates existing Unity Catalog Volume files from flat structure to production-style hierarchical structure.

## Structure

**Before (Flat):**
```
/Volumes/catalog/schema/volume/
  └── 202512...usertable...parquet
```

**After (Hierarchical):**
```
/Volumes/catalog/schema/volume/
  └── parquet/
      └── defaultdb/
          └── public/
              └── existing-parquet-files/
                  └── 202512...usertable...parquet
```

## Usage

1. Update configuration in Cell 2
2. Run Cell 3 to define migration function
3. Run Cell 4 to execute migration
4. (Optional) Run Cell 5 to verify


In [ ]:
# ==============================================================================
# Configuration - Update these for your setup
# ==============================================================================

FORMAT = "parquet"  # or "json"
CATALOG = "defaultdb"
SCHEMA = "public"
SCENARIO = "existing-parquet-files"  # Name for existing data

# Volume base path
VOLUME_BASE = "dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files"

print("Configuration:")
print(f"  Format:   {FORMAT}")
print(f"  Catalog:  {CATALOG}")
print(f"  Schema:   {SCHEMA}")
print(f"  Scenario: {SCENARIO}")
print(f"  Volume:   {VOLUME_BASE}")
print()
print(f"Target path will be:")
print(f"  {VOLUME_BASE}/{FORMAT}/{CATALOG}/{SCHEMA}/{SCENARIO}/")



Configuration:
  Format:   parquet
  Catalog:  defaultdb
  Schema:   public
  Scenario: existing-parquet-files
  Volume:   dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files

Target path will be:
  dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/


In [ ]:
# ==============================================================================
# Migration Function
# ==============================================================================

def migrate_to_hierarchy():
    """Migrate files from flat structure to hierarchical structure."""
    
    # Source: Flat structure (old way)
    source_path = f"{VOLUME_BASE}/"
    
    # Target: Hierarchical structure (production way)
    target_path = f"{VOLUME_BASE}/{FORMAT}/{CATALOG}/{SCHEMA}/{SCENARIO}/"
    
    print("=" * 80)
    print("VOLUME MIGRATION - Production Hierarchy")
    print("=" * 80)
    print(f"Format:   {FORMAT}")
    print(f"Catalog:  {CATALOG}")
    print(f"Schema:   {SCHEMA}")
    print(f"Scenario: {SCENARIO}")
    print()
    print(f"Source (flat):         {source_path}")
    print(f"Target (hierarchical): {target_path}")
    print("=" * 80)
    print()
    
    # Create target directory with full hierarchy
    print("📁 Creating target directory...")
    dbutils.fs.mkdirs(target_path)
    print(f"   ✅ Created: {target_path}")
    print()
    
    # Determine file extension
    file_ext = '.parquet' if FORMAT == 'parquet' else '.ndjson'
    
    # List source files
    print(f"📋 Listing source files ({file_ext})...")
    try:
        files = dbutils.fs.ls(source_path)
    except Exception as e:
        print(f"   ❌ Error listing files: {e}")
        return
    
    # Filter files by extension
    target_files = [f for f in files if f.name.endswith(file_ext) and not f.name.endswith('/')]
    
    if not target_files:
        print(f"   ⚠️  No {file_ext} files found in source directory")
        return
    
    print(f"   Found: {len(target_files)} files")
    print()
    
    # Move files
    print(f"📦 Moving files to hierarchical structure...")
    print()
    
    moved_count = 0
    skipped_count = 0
    
    for file in target_files:
        source = file.path
        target = target_path + file.name
        
        try:
            dbutils.fs.mv(source, target)
            moved_count += 1
            print(f"  ✅ {file.name}")
        except Exception as e:
            skipped_count += 1
            print(f"  ⚠️  {file.name}: {e}")
    
    # Summary
    print()
    print("=" * 80)
    print("MIGRATION COMPLETE")
    print("=" * 80)
    print(f"✅ Moved:   {moved_count} files")
    if skipped_count > 0:
        print(f"⚠️  Skipped: {skipped_count} files")
    print()
    print(f"📂 Files now at:")
    print(f"   {target_path}")
    print()
    print(f"📝 Update your notebooks to use:")
    print(f"   VOLUME_PATH = '{VOLUME_BASE}/{FORMAT}/{CATALOG}/{SCHEMA}/{SCENARIO}'")
    print()
    print("=" * 80)
    print()
    print("🔍 Verify migration:")
    print(f"   dbutils.fs.ls('{target_path}')")
    print()

print("✅ Migration function defined")



✅ Migration function defined


In [ ]:
# ==============================================================================
# Run Migration
# ==============================================================================

# Run the migration
migrate_to_hierarchy()



VOLUME MIGRATION - Production Hierarchy
Format:   parquet
Catalog:  defaultdb
Schema:   public
Scenario: existing-parquet-files

Source (flat):         dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/
Target (hierarchical): dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/

📁 Creating target directory...
   ✅ Created: dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/

📋 Listing source files (.parquet)...
   Found: 11 files

📦 Moving files to hierarchical structure...

  ✅ 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000000-usertable+fam_0_ycsb_key-4.parquet
  ✅ 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000001-usertable+fam_10_field9-4.parquet
  ✅ 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000002-usertable+fam_1_field0-4.parquet
  ✅ 202512191714242809831900

In [ ]:
# ==============================================================================
# Verify Migration (Optional)
# ==============================================================================

# Check files in new location
target_path = f"{VOLUME_BASE}/{FORMAT}/{CATALOG}/{SCHEMA}/{SCENARIO}/"

print(f"Verifying files at: {target_path}")
print()

try:
    files = dbutils.fs.ls(target_path)
    print(f"✅ Found {len(files)} files:")
    print()
    
    # Show first 10 files
    for file in files[:10]:
        print(f"  • {file.name}")
    
    if len(files) > 10:
        print(f"  ... and {len(files) - 10} more files")
    
except Exception as e:
    print(f"❌ Error: {e}")



Verifying files at: dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/existing-parquet-files/parquet/defaultdb/public/existing-parquet-files/

✅ Found 11 files:

  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000000-usertable+fam_0_ycsb_key-4.parquet
  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000001-usertable+fam_10_field9-4.parquet
  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000002-usertable+fam_1_field0-4.parquet
  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000003-usertable+fam_2_field1-4.parquet
  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000004-usertable+fam_3_field2-4.parquet
  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000005-usertable+fam_4_field3-4.parquet
  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000006-usertable+fam_5_field4-4.parquet
  • 202512191714242809831900000000000-d340a6adc87c635b-1-375-00000007-usertable+fam_6_field5-4.parquet
  • 2

## Next Steps

After migration, update your notebooks to use the new hierarchical path:

### Update Volume Path

```python
# Old (flat structure)
VOLUME_PATH = "dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files"

# New (hierarchical structure)
VOLUME_PATH = "dbfs:/Volumes/main/robert_lee_cockroachdb/parquet_files/parquet/defaultdb/public/existing-parquet-files"
```

### Example Notebook Usage

```python
# In load_parquet_with_merge.ipynb or test_cdc_scenario.ipynb

# Cell 1 - Configuration
VOLUME_PATH = f"{base_volume}/parquet/defaultdb/public/existing-parquet-files"
SOURCE_TABLE = "usertable"
PRIMARY_KEY_COLUMNS = ["ycsb_key"]

# Cell 2 - Run automated test
from cockroachdb import load_and_merge_cdc_to_delta

result = load_and_merge_cdc_to_delta(
    source_table=SOURCE_TABLE,
    volume_path=VOLUME_PATH,
    target_table_path=TARGET_TABLE_PATH,
    crdb_config=CRDB_CONFIG,
    catalog="defaultdb",
    schema="public",
    clear_checkpoint=True,
    verify=True
)
```

## Benefits of Hierarchical Structure

✅ **Organized by format** - Easy to separate parquet from json files  
✅ **Multi-catalog support** - Can organize multiple databases  
✅ **Production-ready** - Same structure as production changefeeds  
✅ **Self-documenting** - Path reveals format/catalog/schema/scenario
